# Hehehehe

In [1]:
print("a")

a


In [ ]:
import pandas as pd
from sklearn.ensemble import RandomForestClassifier

# 1. FUNKTION DEFINIEREN (Muss zwingend vor dem Aufruf stehen)
def get_data_as_of(df: pd.DataFrame, as_of_date: str) -> pd.DataFrame:
    """Filtert den historischen Datensatz für einen exakten Stichtag."""
    target_date = pd.to_datetime(as_of_date)
    mask = (df['valid_from'] <= target_date) & (df['valid_to'] >= target_date)
    return df[mask].reset_index(drop=True)

# 2. DATENBASIS ERSTELLEN
data_med = [
    # Patient 1: Entwickelt später Krankheit D
    {"id": 1, "alter": 55, "diagnose_a": 0, "diagnose_b": 0, "krankenhaus": 0, "krankheit_d": 0, "valid_from": "2023-01-01", "valid_to": "2024-05-31"},
    {"id": 1, "alter": 56, "diagnose_a": 1, "diagnose_b": 0, "krankenhaus": 1, "krankheit_d": 0, "valid_from": "2024-06-01", "valid_to": "2024-11-30"},
    {"id": 1, "alter": 56, "diagnose_a": 1, "diagnose_b": 1, "krankenhaus": 1, "krankheit_d": 1, "valid_from": "2024-12-01", "valid_to": "2099-12-31"}, 
    
    # Patient 2: Bleibt gesund
    {"id": 2, "alter": 40, "diagnose_a": 0, "diagnose_b": 0, "krankenhaus": 0, "krankheit_d": 0, "valid_from": "2022-01-01", "valid_to": "2025-06-30"},
    {"id": 2, "alter": 43, "diagnose_a": 0, "diagnose_b": 0, "krankenhaus": 0, "krankheit_d": 0, "valid_from": "2025-07-01", "valid_to": "2099-12-31"},
    
    # Patient 3: Hat Diagnose A, aber kein D
    {"id": 3, "alter": 62, "diagnose_a": 1, "diagnose_b": 0, "krankenhaus": 0, "krankheit_d": 0, "valid_from": "2023-05-01", "valid_to": "2099-12-31"},
    
    # Patient 4: Entwickelt Krankheit D
    {"id": 4, "alter": 58, "diagnose_a": 1, "diagnose_b": 0, "krankenhaus": 0, "krankheit_d": 0, "valid_from": "2023-01-01", "valid_to": "2024-08-15"},
    {"id": 4, "alter": 59, "diagnose_a": 1, "diagnose_b": 1, "krankenhaus": 1, "krankheit_d": 1, "valid_from": "2024-08-16", "valid_to": "2099-12-31"}
]

df_med_hist = pd.DataFrame(data_med)
df_med_hist['valid_from'] = pd.to_datetime(df_med_hist['valid_from'])
df_med_hist['valid_to'] = pd.to_datetime(df_med_hist['valid_to'])

# 3. DATEN FÜR DAS TRAINING EXTRAHIEREN (Point-in-Time)
stichtag_training = "2024-07-01"
df_features = get_data_as_of(df_med_hist, stichtag_training)

df_latest = get_data_as_of(df_med_hist, "2099-12-31")[['id', 'krankheit_d']]
df_train = pd.merge(df_features, df_latest, on='id', suffixes=('', '_future'))

# 4. MODELL TRAINIEREN
X = df_train[['alter', 'diagnose_a', 'diagnose_b', 'krankenhaus']]
y = df_train['krankheit_d_future'] 

model = RandomForestClassifier(random_state=42, n_estimators=10)
model.fit(X, y)
print("Modell erfolgreich auf historisierten Daten trainiert.\n")

# 5. PROGNOSE ERSTELLEN
neuer_patient = pd.DataFrame([{
    "alter": 57, 
    "diagnose_a": 1, 
    "diagnose_b": 0, 
    "krankenhaus": 1
}])

wahrscheinlichkeit = model.predict_proba(neuer_patient)[0][1] 

print(f"--- Risiko-Prognose für neuen Patienten ---")
print(f"Patientendaten: Alter 57, Diagnose A vorhanden, Krankenhausaufenthalt.")
print(f"Wahrscheinlichkeit für zukünftigen Ausbruch von Krankheit D: {wahrscheinlichkeit * 100:.1f} %")

Modell erfolgreich auf historisierten Daten trainiert.

--- Risiko-Prognose für neuen Patienten ---
Patientendaten: Alter 57, Diagnose A vorhanden, Krankenhausaufenthalt.
Wahrscheinlichkeit für zukünftigen Ausbruch von Krankheit D: 80.0 %


: 